# Borealis-27b at full bf16 — NorSumm + NorQuAD + QGEval

Runs a Borealis-27b variant at its **native bf16 precision — nothing quantized** — over **three** benchmarks:

| Benchmark | Task | Metric | Size |
|---|---|---|---|
| **NorSumm** | summarization | ROUGE-1/2/L | 33 articles |
| **NorQuAD** | extractive QA | Exact Match / F1 | 300 questions |
| **QGEval** | answer-conditioned question generation | 7 dimensions (Fluency, Clarity, Conciseness, Relevance, Consistency, Answerability, Answer Consistency) | 200 passages |

## Which model

Two exist, and they are **not** the same thing:

| Model | What it is | bf16 size |
|---|---|---|
| `NbAiLab/borealis-27b` | instruction-tuned **full release** | 51.1 GiB |
| `NbAiLab/borealis-27b-instruct-preview` | earlier **preview** — its own card calls it *"an experiment ... pre-release quality"*, tuned from `google/gemma-3-27b-it` | 53.7 GiB |

The preview is **not** a newer or better model. Output filenames carry the model name, so the two produce separate rows rather than overwriting each other.

**Run this whole notebook once per model** (set `MODEL_ID` in cell 5b, run all three benchmark cells, download, repeat with the other `MODEL_ID`) to get both variants into the comparison.

## Before you start

**Runtime → Change runtime type → `A100 GPU` + `High-RAM`.**

On **Colab Pro+**, also enable **background execution**. All three benchmarks off one model load takes several hours.

Neither model fits on Colab's 40 GB A100, so accelerate splits it across GPU and CPU RAM. Everything stays bf16; the offloaded layers just make generation slow.

**The model is loaded once and reused for all three benchmarks** — do not re-run the load cell between them.

## 1. Check what card you were assigned

Run this first. If it says anything other than A100, use **Runtime → Disconnect and delete runtime** and try again — Colab assigns cards from a pool.

In [ ]:
import torch, psutil

WEIGHTS = 53.7  # GiB — larger of the two variants; cell 5b prints the exact figure

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > A100 GPU + High-RAM.')

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram = psutil.virtual_memory().total / 1024**3
gpu_budget = vram - 6
spill = max(0.0, WEIGHTS - gpu_budget)

print(f'GPU: {name} ({vram:.1f} GiB)')
print(f'RAM: {ram:.1f} GiB')
print(f'Weights: {WEIGHTS} GiB bf16\n')

if vram < 30:
    print(f'STOP. {name} is too small. Disconnect and delete the runtime, then retry for an A100.')
elif spill == 0:
    print('Excellent — fully GPU-resident. Expect ~10-20 minutes for all 33 articles.')
elif spill > ram - 8:
    print(f'STOP. {spill:.1f} GiB must spill to RAM but only {ram-8:.1f} GiB is usable.')
    print('Switch to a High-RAM runtime.')
else:
    print(f'OK: {gpu_budget:.1f} GiB on GPU, {spill:.1f} GiB offloaded to RAM.')
    print('Expect ~1-3 tok/s, roughly 1.5-4 hours for all 33 articles.')
    if ram < 60:
        print('\nNOTE: this looks like standard-RAM. High-RAM is strongly recommended.')

## 2. Install dependencies

`bitsandbytes` is deliberately **not** installed — this run is unquantized.

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub pandas pyarrow
import transformers, accelerate
print('transformers', transformers.__version__, '| accelerate', accelerate.__version__)

## 3. Mount Drive (recommended)

The script checkpoints after **every article**. Writing those checkpoints to Drive means a disconnect costs one article instead of the whole run. Skip this cell and it falls back to local storage, which is wiped when the runtime dies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# One subdirectory per benchmark — all three write the same filename for a given model.
!mkdir -p /content/drive/MyDrive/borealis_bench/norsumm /content/drive/MyDrive/borealis_bench/qa /content/drive/MyDrive/borealis_bench/qgeval

## 4. Get the benchmark code

Public repo, so no token needed.

In [ ]:
!git clone -q --branch claude/norquad-norsumm-benchmarks-9qhyke \
    https://github.com/rymarinelli/embedding.git /content/embedding
%cd /content/embedding/benchmarks/scripts
!ls generate_summaries_borealis_bf16_colab.py generate_qa_answers_borealis_bf16_colab.py generate_questions_qgeval_borealis_colab.py
!ls ../data/qgeval_sample.json

## 5. (Optional) Hugging Face login

`NbAiLab/borealis-27b` is not gated, so this is only useful for download rate limits. Skip it if you like.

In [ ]:
# from huggingface_hub import login
# login(token='hf_...')   # never commit this

## 5b. Choose the model

Edit `MODEL_ID` here. Everything downstream — weight-size checks, memory budgets, output filenames and result labels — follows from it.

In [ ]:
import generate_summaries_borealis_bf16_colab as run

# 'NbAiLab/borealis-27b'                  -> instruction-tuned full release (51.1 GiB)
# 'NbAiLab/borealis-27b-instruct-preview' -> earlier preview, pre-release quality (53.7 GiB)
run.MODEL_ID = 'NbAiLab/borealis-27b-instruct-preview'
run.RUN_LABEL, run.OUT_NAME, run.OUT_PATH = run.derive_paths(run.MODEL_ID)

print('model      :', run.MODEL_ID)
print('bf16 weights:', f'{run.model_weight_gib():.1f} GiB')
print('will score as:', run.RUN_LABEL)

## 6. Run

Downloads ~51 GiB, then generates. Watch the **first article's timing**: if it takes more than ~8 minutes, the offload is thrashing and the run won't finish in a sitting — stop and confirm you got High-RAM.

`REPETITION_PENALTY` is left at **1.0**, matching the original quantized run, so this isolates the effect of precision alone. Only change it for a separate second run.

In [ ]:
# Load once, keep the handles — the QA benchmark below reuses them.
run.preflight()
processor, model = run.load_model()

print('\nrepetition_penalty =', run.REPETITION_PENALTY, '| temperature =', run.TEMPERATURE)
print('checkpointing to  =', run.OUT_PATH)

## 7. Benchmark A — NorSumm (summarization)

33 articles, ROUGE-1/2/L. Watch the **first article's timing**: more than ~8 minutes means the offload is thrashing — stop and confirm you got High-RAM.

Checkpoints after every article, so a disconnect costs one article. Re-run this cell to resume.

In [ ]:
run.main(model=model, processor=processor)

## 8. Benchmark B — NorQuAD (extractive QA)

300 questions, Exact Match / token-F1 — the numbers in Table 2. **Reuses the model already loaded above**, so there is no second 51 GiB download.

Decoding is greedy with `max_new_tokens=64`, matching `generate_qa_answers_local_gguf.py` so the EM/F1 are comparable with the API models. Note this differs from the summarization run, which matched its own baseline at temperature 0.2.

300 short answers on offloaded weights still takes a while, but each is far shorter than a summary. Checkpoints every question.

In [ ]:
import generate_qa_answers_borealis_bf16_colab as qa

qa.main(model=model, processor=processor)

## 9. Benchmark C — QGEval (answer-conditioned question generation)

200 passages. Generates one answer-conditioned question per passage using the exact same prompt as the OpenRouter models and the local-GGUF path, so it drops straight into the multi-model comparison — `score_qgeval_dimensions_multi.py` and `report_qgeval_multi.py` in the main repo.

**Reuses the model already loaded above.** Output goes to `results/qgeval_questions/<model-name>.json` — that's a *different shape and location* from the NorSumm/QA outputs, so it won't collide with them.

Judging (the actual 7-dimension scores) happens back in the main repo with an OpenRouter judge, not here — this cell only generates the questions.

In [ ]:
import generate_questions_qgeval_borealis_colab as qg

qg.main(model=model, processor=processor)

## 10. Download all result files

| File | Goes in | Then run |
|---|---|---|
| `<model>-bf16-full.json` (summaries) | `benchmarks/results/generated_summaries/` | `score_summaries.py` |
| `<model>-bf16-full.json` (QA) | `benchmarks/results/qa_answers/` | `score_qa.py` |
| `<model-name>.json` (QGEval) | `benchmarks/results/qgeval_questions/` | `score_qgeval_dimensions_multi.py` then `report_qgeval_multi.py` |

The QGEval file's name is just the bare model name (e.g. `borealis-27b-instruct-preview.json`), not `-bf16-full` — the multi-model QGEval scripts key on the plain model name across every generation path (OpenRouter, local GGUF, this notebook).

In [ ]:
from google.colab import files
import os, json

base = '/content/drive/MyDrive/borealis_bench'
for task in ('norsumm', 'qa', 'qgeval'):
    d = os.path.join(base, task)
    if not os.path.isdir(d):
        print(f'{task}: nothing yet'); continue
    for f in sorted(os.listdir(d)):
        if f.endswith('.json'):
            path = os.path.join(d, f)
            print(f'{task}: {f} ({len(json.load(open(path)))} records)')
            files.download(path)